In [ ]:
import pandas as pd

# load just the datasets q01 needs.
factor = 10
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # Use GPU parse_dates to avoid post-read conversions
    df = pd.read_csv(
        data_path,
        sep="|",
        storage_options=storage_options,
        parse_dates=["L_SHIPDATE", "L_RECEIPTDATE", "L_COMMITDATE"]
    )
    print(df.columns)
    return df

lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_supplier(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/supplier.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df

supplier = load_supplier(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
### cell 0 ###

# Optimized GPU workflow using cudf.pandas
# Define date bounds as strings to push comparisons to the GPU
date_start = "1996-01-01"
date_end = "1996-04-01"

# Filter, compute revenue parts, aggregate and pick top supplier in one pipeline
top_revenue = (
    lineitem
    .loc[
        (lineitem.L_SHIPDATE >= date_start) & (lineitem.L_SHIPDATE < date_end),
        ["L_SUPPKEY", "L_EXTENDEDPRICE", "L_DISCOUNT"]
    ]
    .assign(REVENUE_PARTS=lambda df: df.L_EXTENDEDPRICE * (1 - df.L_DISCOUNT))
    .groupby("L_SUPPKEY")["REVENUE_PARTS"].sum()
    .reset_index()
    .nlargest(1, "REVENUE_PARTS")
    .rename(columns={"L_SUPPKEY": "S_SUPPKEY", "REVENUE_PARTS": "TOTAL_REVENUE"})
)

# Join with supplier details and select final columns
total = (
    top_revenue
    .merge(
        supplier[["S_SUPPKEY", "S_NAME", "S_ADDRESS", "S_PHONE"]],
        on="S_SUPPKEY"
    )
    [["S_SUPPKEY", "S_NAME", "S_ADDRESS", "S_PHONE", "TOTAL_REVENUE"]]
)